In [24]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai
import operator

load_dotenv()
import os

In [25]:
#creating a rule or format for out[ut and asking llm to give output in that format. It is done using pydantic

class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay")
    score: int = Field(ge=0, le=10, description="Score out of 10 based on content, structure, and language")



In [ ]:
# Initialize the GenAI client

client = genai.Client(
    vertexai=True, 
    project=os.getenv('GEMINI_API_KEY'), 
    location='us-central1'
)

GEMINI_SETTINGS = {
    "model": "gemini-2.0-flash",
    "config": {
        'response_mime_type': 'application/json',
        'response_schema': EvaluationSchema,
    }
}

In [39]:
essay = '''The Architecture of Wonder: Why Curiosity is the Engine of Progress
In the traditional nursery rhyme, curiosity is the culprit that "killed the cat." This cautionary phrase suggests that inquisitiveness is a dangerous trait, one that leads to unnecessary risk and trouble. However, history and human experience tell a vastly different story. Rather than a liability, curiosity is the primary architect of human civilization. It is the restless intellectual hunger that transforms a simple observation into a scientific breakthrough, an empathetic connection, or a masterpiece of art.
At its core, curiosity is the refusal to accept the world at face value. It is the "what if" and the "why" that separates humans from a purely instinctual existence. In the realm of science, curiosity is the catalyst for discovery. When Isaac Newton watched an apple fall, it was not merely a physical event; it was a question. When Alexander Fleming noticed mold growing in a petri dish, curiosity prevented him from simply throwing it away, leading instead to the discovery of penicillin. Without this drive to look closer at the "accidents" of life, we would still be living in the shadows of the Dark Ages, deprived of the medical and technological marvels we now take for granted.
Beyond the laboratory, curiosity serves as a vital bridge for social and emotional intelligence. To be curious about another person is the first step toward empathy. When we are curious about cultures, histories, and perspectives different from our own, prejudice begins to dissolve. It is impossible to truly hate a person whose story you are genuinely interested in learning. By asking questions rather than making assumptions, we replace the walls of judgment with the windows of understanding, making curiosity an essential tool for global harmony.
On a personal level, curiosity is the secret to lifelong vitality. Neuroscientists have found that the brain’s chemistry changes when we are curious; it releases dopamine, the "reward" chemical, making the process of learning inherently pleasurable. Those who maintain a "beginner’s mind"—a state of constant wondering—are more resilient in the face of change. They view challenges not as insurmountable walls, but as puzzles waiting to be solved. In an era where information is becoming obsolete at an increasing rate, the ability to remain curious is more valuable than any specific degree or technical skill.
In conclusion, curiosity is the most powerful resource we possess. It is the fuel for innovation, the foundation of empathy, and the heartbeat of personal growth. While it may lead us into the unknown, it is only in the unknown that progress is made. To live a rich and meaningful life, one must not fear the "cat-killing" risks of wonder, but rather embrace them, for a life without curiosity is a life standing still.'''

In [45]:
prompt = f"Evaluate the following essay on a scale of 0 to 10 ."



In [46]:

response = client.models.generate_content(
    contents=essay + "\n\n" + prompt,
    **GEMINI_SETTINGS
)

In [47]:
structured_data = response.parsed
# print(structured_data)

print(f"Score: {structured_data.score}")
print(f"Feedback: {structured_data.feedback}")

Score: 9
Feedback: This is a well-written and insightful essay that effectively argues for the importance of curiosity. The essay presents a clear thesis and supports it with compelling examples from science, social interaction, and personal development. The writing is engaging and uses vivid language to illustrate its points. The structure is logical, with a clear introduction, body paragraphs, and conclusion. The essay also effectively refutes the traditional negative view of curiosity. Overall, this is a strong and persuasive piece of writing.


In [48]:
# Defining the state

class UPSCState(TypedDict):
    essay : str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int],operator.add]
    avg_score: float

     

In [50]:
def evaluate_language(state : UPSCState):
    prompt = "Evaluate the following essay specifically for language quality on a scale of 0 to 10. Provide detailed feedback as well."

    output = client.models.generate_content(
    contents=state['essay'] + "\n\n" + prompt,
    **GEMINI_SETTINGS)

    return {'language_feedback': output.parsed.feedback,'individual_scores':[output.parsed.score]}

In [51]:
def evaluate_analysis(state : UPSCState):
    prompt = "Evaluate the following essay specifically for an analysis quality on a scale of 0 to 10. Provide detailed feedback as well."

    output = client.models.generate_content(
    contents=state['essay'] + "\n\n" + prompt,
    **GEMINI_SETTINGS)

    return {'analysis_feedback': output.parsed.feedback,'individual_scores':[output.parsed.score]}

In [52]:
def evaluate_thoughts(state : UPSCState):
    prompt = "Evaluate the following essay specifically for thought quality on a scale of 0 to 10. Provide detailed feedback as well."

    output = client.models.generate_content(
    contents=state['essay'] + "\n\n" + prompt,
    **GEMINI_SETTINGS)

    return {'clarity_feedback': output.parsed.feedback,'individual_scores':[output.parsed.score]}

In [60]:
def final_evaluation(state:UPSCState):
    prompt = f'based on the following feedbacks:\nLanguage Feedback: {state["language_feedback"]}\nAnalysis Feedback: {state["analysis_feedback"]}\nClarity Feedback: {state["clarity_feedback"]}\nProvide an overall feedback for the essay and also provide an average score out of 10.'

    overall_feedback = client.models.generate_content(
    contents=prompt,
    model="gemini-2.0-flash",)

    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback': overall_feedback.text,'avg_score': avg_score}



In [66]:
graph = StateGraph(UPSCState)

In [67]:
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thoughts',evaluate_thoughts)
graph.add_node('final_evaluation',final_evaluation)

In [68]:
# Edges

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thoughts')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')  
graph.add_edge('evaluate_thoughts','final_evaluation')

graph.add_edge('final_evaluation',END)



In [69]:
workflow = graph.compile()

In [70]:
#Displaying the graph
print(workflow.get_graph().draw_ascii())

                                    +-----------+                                      
                                   *| __start__ |**                                    
                              ***** +-----------+  *****                               
                        ******            *             ******                         
                   *****                  *                   *****                    
                ***                       *                        ***                 
+-------------------+           +-------------------+           +-------------------+  
| evaluate_analysis |           | evaluate_language |           | evaluate_thoughts |  
+-------------------+***        +-------------------+         **+-------------------+  
                        ******            *             ******                         
                              *****       *        *****                               
                                

In [72]:
essay1 = ''' 
माझा लाडका देव: गणपती बाप्पा
प्रस्तावना:
"गणपती बाप्पा मोरया, मंगलमूर्ती मोरया!" हा जयघोष ऐकला की प्रत्येक मराठी माणसाचे मन आनंदाने भरून येते. गणपती बाप्पा हा केवळ महाराष्ट्राचाच नव्हे, तर साऱ्या जगाचा लाडका देव आहे. गणपतीला आपण बुद्धीची देवता, कलांचा अधिपती आणि 'विघ्नहर्ता' मानतो. कोणत्याही शुभ कार्याची सुरुवात करताना आपण पहिले नमन गणपतीला करतो.
बाप्पाचे मोहक रूप:
गणपती बाप्पाचे रूप अतिशय गोजिरवाणे आणि आकर्षक आहे. त्याचे हत्तीचे तोंड, लांब सोंड, मोठे कान, एकदंत आणि मोठे पोट हे त्याचे वेगळेपण दर्शवते. बाप्पाच्या मोठ्या कानांचा अर्थ असा की त्याने प्रत्येकाचे गाऱ्हाणे ऐकून घ्यावे आणि छोट्या डोळ्यांचा अर्थ असा की त्याने प्रत्येक गोष्टीचे सूक्ष्म निरीक्षण करावे. बाप्पाचे वाहन छोटासा उंदीर आहे, जे हे शिकवते की कोणतेही काम लहान किंवा मोठे नसते.
गणेशोत्सव:
भाद्रपद महिन्यातील 'गणेश चतुर्थी'ला बाप्पाचे आगमन होते. हा सण महाराष्ट्रात अत्यंत उत्साहात साजरा केला जातो. घराघरांत आणि सार्वजनिक मंडळांमध्ये बाप्पाची प्रतिष्ठापना केली जाते. सजावट, रोषणाई आणि फुलांच्या सुगंधाने वातावरण मंगलमय होते. बाप्पाला प्रिय असलेले 'मोदक' घराघरात बनवले जातात. सकाळी आणि संध्याकाळी होणारी आरती आणि मंत्रजापामुळे मनात शांतता आणि भक्ती निर्माण होते.
सामाजिक महत्त्व:
सार्वजनिक गणेशोत्सवाची सुरुवात लोकमान्य टिळकांनी केली. या सणाचे मुख्य उद्दिष्ट लोकांना एकत्र आणणे आणि समाजात एकजूट निर्माण करणे हे होते. आजही या सणामुळे विविध जाती-धर्माचे लोक एकत्र येतात, सांस्कृतिक कार्यक्रम होतात आणि सामाजिक विषयांवर जनजागृती केली जाते.
निरोप (विसर्जन):
दहा दिवसांच्या पाहुणचारानंतर 'अनंत चतुर्दशी'ला बाप्पाला निरोप देण्याची वेळ येते. मोठ्या मिरवणुकीने बाप्पाचे विसर्जन केले जाते. "गणपती बाप्पा मोरया, पुढच्या वर्षी लवकर या!" अशा घोषणा देताना भक्तांचे डोळे पाणावतात. बाप्पा जाताना आपल्यासोबत सर्व दु:ख घेऊन जातो आणि जाताना सुख-समृद्धीचा आशीर्वाद देऊन जातो.
निष्कर्ष:
गणपती बाप्पा हा आनंदाचे प्रतीक आहे. तो आपल्याला संकटांशी लढण्याची शक्ती आणि बुद्धी देतो. बाप्पाच्या आगमनाने येणारा उत्साह आपल्या जीवनात नवी उमेद आणि चैतन्य निर्माण करतो. म्हणूनच गणपती बाप्पा हा सर्वांचाच अतिशय जवळचा आणि लाडका सखा आहे.'''

In [75]:
initial_state = {'essay': essay1}

workflow.invoke(initial_state)

{'essay': ' \nमाझा लाडका देव: गणपती बाप्पा\nप्रस्तावना:\n"गणपती बाप्पा मोरया, मंगलमूर्ती मोरया!" हा जयघोष ऐकला की प्रत्येक मराठी माणसाचे मन आनंदाने भरून येते. गणपती बाप्पा हा केवळ महाराष्ट्राचाच नव्हे, तर साऱ्या जगाचा लाडका देव आहे. गणपतीला आपण बुद्धीची देवता, कलांचा अधिपती आणि \'विघ्नहर्ता\' मानतो. कोणत्याही शुभ कार्याची सुरुवात करताना आपण पहिले नमन गणपतीला करतो.\nबाप्पाचे मोहक रूप:\nगणपती बाप्पाचे रूप अतिशय गोजिरवाणे आणि आकर्षक आहे. त्याचे हत्तीचे तोंड, लांब सोंड, मोठे कान, एकदंत आणि मोठे पोट हे त्याचे वेगळेपण दर्शवते. बाप्पाच्या मोठ्या कानांचा अर्थ असा की त्याने प्रत्येकाचे गाऱ्हाणे ऐकून घ्यावे आणि छोट्या डोळ्यांचा अर्थ असा की त्याने प्रत्येक गोष्टीचे सूक्ष्म निरीक्षण करावे. बाप्पाचे वाहन छोटासा उंदीर आहे, जे हे शिकवते की कोणतेही काम लहान किंवा मोठे नसते.\nगणेशोत्सव:\nभाद्रपद महिन्यातील \'गणेश चतुर्थी\'ला बाप्पाचे आगमन होते. हा सण महाराष्ट्रात अत्यंत उत्साहात साजरा केला जातो. घराघरांत आणि सार्वजनिक मंडळांमध्ये बाप्पाची प्रतिष्ठापना केली जाते. सजावट, रोषणाई आणि फुलांच्या सुगंधाने वा